In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Freight_Rate_ML_Assessment/train-test.csv")

In [ ]:
print(df.head())

print(df.shape)

print(df.info())

print(df.describe(include='all'))

     load_id        pickup      delivery  pickup_lat  pickup_lon  \
0  TR-000001      Richmond     Baltimore    38.09122   -76.78906   
1  TR-000002      Richmond  Philadelphia    38.09122   -76.78906   
2  TR-000003  Philadelphia     Green Bay    39.22317   -72.96710   
3  TR-000004      Hartford       Atlanta    39.55328   -72.18051   
4  TR-000005        Dallas     Nashville    31.83025   -94.38343   

   delivery_lat  delivery_lon  distance equipment   weight        date  \
0      38.16908     -72.74564     274.3   Dry Van  30658.0  2025-01-01   
1      39.22317     -72.96710     280.5    Reefer  17555.0  2025-01-01   
2      44.30296     -87.52871     967.8   Dry Van  31721.0  2025-01-01   
3      34.84933     -86.28940     965.4   Dry Van  32333.0  2025-01-01   
4      35.29479     -88.08915     541.9    Reefer  35183.0  2025-01-01   

   market_index  quote_signal  posted_rate  
0       0.95684       2.39595       645.41  
1       0.97623       2.43355       679.97  
2       1.0

In [ ]:
print(df.isnull().sum())

load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
posted_rate     0
dtype: int64


In [ ]:
df["weight"] = df["weight"].fillna(df["weight"].median())

df["market_index"] = df["market_index"].fillna(df["market_index"].median())

In [ ]:
df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

df.drop("date",axis=1,inplace=True)

In [ ]:
X = df.drop(
    columns=[
        "posted_rate",
        "load_id"
    ]
)

y = df["posted_rate"]

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print(categorical_features)

print(numeric_features)

['pickup', 'delivery', 'equipment']
['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'market_index', 'quote_signal', 'year', 'month', 'day', 'day_of_week']


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42
)

In [ ]:
rf_model = Pipeline(

    steps=[

        ("preprocessor",preprocessor),

        ("model",

         RandomForestRegressor(

             random_state=42,

             n_estimators=200

         )

        )

    ]

)

rf_model.fit(X_train,y_train)

rf_pred=rf_model.predict(X_test)

print("Random Forest")

print("MAE :",mean_absolute_error(y_test,rf_pred))

print("RMSE :",np.sqrt(mean_squared_error(y_test,rf_pred)))

print("R2 :",r2_score(y_test,rf_pred))

KeyboardInterrupt: 

In [ ]:
gb_model=Pipeline(

    steps=[

        ("preprocessor",preprocessor),

        ("model",

         GradientBoostingRegressor(

             random_state=42

         )

        )

    ]

)

gb_model.fit(X_train,y_train)

gb_pred=gb_model.predict(X_test)

print("Gradient Boosting")

print("MAE :",mean_absolute_error(y_test,gb_pred))

print("RMSE :",np.sqrt(mean_squared_error(y_test,gb_pred)))

print("R2 :",r2_score(y_test,gb_pred))

Gradient Boosting
MAE : 107.62451537590475
RMSE : 530.1909677909315
R2 : 0.8685597658493046


In [ ]:
xgb_model=Pipeline(

    steps=[

        ("preprocessor",preprocessor),

        ("model",

         XGBRegressor(

             random_state=42,

             n_estimators=300,

             learning_rate=0.05,

             max_depth=6,

             objective="reg:squarederror"

         )

        )

    ]

)

xgb_model.fit(X_train,y_train)

xgb_pred=xgb_model.predict(X_test)

print("XGBoost")

print("MAE :",mean_absolute_error(y_test,xgb_pred))

print("RMSE :",np.sqrt(mean_squared_error(y_test,xgb_pred)))

print("R2 :",r2_score(y_test,xgb_pred))

XGBoost
MAE : 124.667804454422
RMSE : 554.3121459915271
R2 : 0.8563278919653443


In [ ]:
results=pd.DataFrame({

    "Model":[

        "Random Forest",

        "Gradient Boosting",

        "XGBoost"

    ],

    "MAE":[

        mean_absolute_error(y_test,rf_pred),

        mean_absolute_error(y_test,gb_pred),

        mean_absolute_error(y_test,xgb_pred)

    ],

    "RMSE":[

        np.sqrt(mean_squared_error(y_test,rf_pred)),

        np.sqrt(mean_squared_error(y_test,gb_pred)),

        np.sqrt(mean_squared_error(y_test,xgb_pred))

    ],

    "R2":[

        r2_score(y_test,rf_pred),

        r2_score(y_test,gb_pred),

        r2_score(y_test,xgb_pred)

    ]

})

results.sort_values("R2",ascending=False)

NameError: name 'rf_pred' is not defined

In [ ]:
final_model = Pipeline(

    steps=[

        ("preprocessor",preprocessor),

        ("model",

         GradientBoostingRegressor(

             random_state=42

         )

        )

    ]

)

final_model.fit(X,y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['pickup_lat', 'pickup_lon',
                                                   'delivery_lat',
                                                   'delivery_lon', 'distance',
                                                   'weight', 'market_index',
                                                   'quote_signal', 'year',
                                                   'month', 'day',
                                                   'day_of_week']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['pickup', 'delivery',
                                                   'equipment'])])),
                ('model', GradientBoostingRegressor(random_state=42))])

In [ ]:
# Model for validation.csv (all features)

X_full = df.drop(columns=["posted_rate", "load_id"])
y = df["posted_rate"]

categorical_full = ["pickup", "delivery", "equipment"]

numeric_full = [col for col in X_full.columns if col not in categorical_full]

preprocessor_full = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_full
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_full
        )
    ]
)

validation_model = Pipeline([
    ("preprocessor", preprocessor_full),
    ("model", GradientBoostingRegressor(random_state=42))
])

validation_model.fit(X_full, y)

print("Validation model trained successfully!")

Validation model trained successfully!


In [ ]:
common_features = [
    "pickup",
    "delivery",
    "distance",
    "equipment",
    "weight",
    "year",
    "month",
    "day",
    "day_of_week"
]

X_common = df[common_features]

categorical_common = [
    "pickup",
    "delivery",
    "equipment"
]

numeric_common = [
    "distance",
    "weight",
    "year",
    "month",
    "day",
    "day_of_week"
]

preprocessor_common = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_common
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_common
        )
    ]
)

december_model = Pipeline([
    ("preprocessor", preprocessor_common),
    ("model", GradientBoostingRegressor(random_state=42))
])

december_model.fit(X_common, y)

print("December model trained successfully!")

December model trained successfully!


In [ ]:
validation = pd.read_csv(
    "/content/drive/MyDrive/Freight_Rate_ML_Assessment/validation.csv"
)

In [ ]:
validation["date"] = pd.to_datetime(validation["date"])

validation["year"] = validation["date"].dt.year
validation["month"] = validation["date"].dt.month
validation["day"] = validation["date"].dt.day
validation["day_of_week"] = validation["date"].dt.dayofweek

validation.drop("date", axis=1, inplace=True)

In [ ]:
validation_features = validation.drop(columns=["load_id"])

validation_predictions = validation_model.predict(validation_features)

In [ ]:
submission = pd.read_csv(
    "/content/drive/MyDrive/Freight_Rate_ML_Assessment/validation-predictions-template.csv"
)

submission["predicted_rate"] = validation_predictions

submission.to_csv(
    "/content/drive/MyDrive/Freight_Rate_ML_Assessment/validation_predictions.csv",
    index=False
)

print("validation_predictions.csv created!")

validation_predictions.csv created!


In [ ]:
december = pd.read_csv(
    "/content/drive/MyDrive/Freight_Rate_ML_Assessment/december-chart-inputs.csv"
)

In [ ]:
december["date"] = pd.to_datetime(december["date"])

december["year"] = december["date"].dt.year
december["month"] = december["date"].dt.month
december["day"] = december["date"].dt.day
december["day_of_week"] = december["date"].dt.dayofweek

In [ ]:
december_features = december[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "year",
        "month",
        "day",
        "day_of_week"
    ]
]

december_predictions = december_model.predict(december_features)

In [ ]:
december["predicted_rate"] = december_predictions

december = december[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]
]

december.to_csv(
    "/content/drive/MyDrive/Freight_Rate_ML_Assessment/december_chart_inputs.csv",
    index=False
)

print("December predictions saved!")

December predictions saved!


In [ ]:
!python "/content/drive/MyDrive/Freight_Rate_ML_Assessment/score.py" \
--predictions "/content/drive/MyDrive/Freight_Rate_ML_Assessment/validation_predictions.csv" \
--december-predictions "/content/drive/MyDrive/Freight_Rate_ML_Assessment/december_chart_inputs.csv"

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
